In [10]:
import os
import zipfile
import requests
import pandas as pd

In [1]:
# Download the dataset
url = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"
response = requests.get(url)
with open("hymenoptera_data.zip", "wb") as f:
    f.write(response.content)

# Unzip the dataset
with zipfile.ZipFile("hymenoptera_data.zip", 'r') as zip_ref:
    zip_ref.extractall(".")

# Define data transformations
from torchvision import datasets, transforms

data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

import torch

data_dir = 'hymenoptera_data'
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x]) for x in ['train', 'val']}
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=4, shuffle=True, num_workers=4) for x in ['train', 'val']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [1]:
import os
import zipfile
import requests
import pandas as pd

In [2]:
datas = []
for dataset in ['train', 'val'] : 
    for idx, class_ in enumerate(['bees', 'ants']) : 
        base_path = os.path.join('hymenoptera_data', dataset, class_)
        paths = os.listdir(base_path)
        for path in paths : 
            datas.append(
                {
                    'dir': os.path.join(base_path, path),
                    'file_name': os.path.basename(path),
                    'dataset': dataset, 
                    'class': class_, 
                    'class_id': idx
                }
            )

In [3]:
df = pd.DataFrame(datas)
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   dir        398 non-null    object
 1   file_name  398 non-null    object
 2   dataset    398 non-null    object
 3   class      398 non-null    object
 4   class_id   398 non-null    int64 
dtypes: int64(1), object(4)
memory usage: 15.7+ KB


,dir,file_name,dataset,class,class_id
0,hymenoptera_data/train/bees/2638074627_6b3ae74...,2638074627_6b3ae746a0.jpg,train,bees,0
1,hymenoptera_data/train/bees/507288830_f46e8d4c...,507288830_f46e8d4cb2.jpg,train,bees,0
2,hymenoptera_data/train/bees/2405441001_b06c36f...,2405441001_b06c36fa72.jpg,train,bees,0
3,hymenoptera_data/train/bees/2962405283_22718d9...,2962405283_22718d9617.jpg,train,bees,0
4,hymenoptera_data/train/bees/446296270_d9e8b93e...,446296270_d9e8b93ecf.jpg,train,bees,0


In [4]:
df[df['dataset']=='train']['class'].value_counts()

class
ants    124
bees    121
Name: count, dtype: int64

In [5]:
df[df['dataset']=='val']['class'].value_counts()

class
bees    83
ants    70
Name: count, dtype: int64

In [6]:
df['dataset'].value_counts()

dataset
train    245
val      153
Name: count, dtype: int64

In [7]:
df.to_parquet('hymenoptera_data_annotation.parquet')